In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Check GPU
!nvidia-smi

# Clone CIRI-FS
%cd /content
!rm -rf CIRI-FS

!git clone --branch asal/CiriEXT4 https://github.com/isusbu/CIRI-FS.git

%cd /content/CIRI-FS

# Verify the correct branch
!git branch --show-current

Wed Aug  5 18:28:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q --upgrade \
    "bitsandbytes>=0.46.1" \
    accelerate \
    transformers \
    sentencepiece \
    anthropic \
    "protobuf>=5.29.1,<6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 127.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.2 MB/s eta 0:00:00


In [4]:
import bitsandbytes as bnb
import transformers
import accelerate
import torch

print("bitsandbytes:", bnb.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

bitsandbytes: 0.50.0
transformers: 5.14.1
accelerate: 1.14.0
PyTorch: 2.11.0+cu128
CUDA available: True


In [5]:
!find /content/drive/MyDrive -type d | grep "EXT4_XML_Dataset"

/content/drive/MyDrive/Dataset/EXT4_XML_Dataset
/content/drive/MyDrive/Dataset/EXT4_XML_Dataset/ext4_CPD
/content/drive/MyDrive/Dataset/EXT4_XML_Dataset/ext4_CPD/correct
/content/drive/MyDrive/Dataset/EXT4_XML_Dataset/ext4_CPD/erroneous
/content/drive/MyDrive/Dataset/EXT4_XML_Dataset/Groundtruth
/content/drive/MyDrive/Dataset/EXT4_XML_Dataset/ext4_CCD
/content/drive/MyDrive/Dataset/EXT4_XML_Dataset/ext4_CCD/erroneous
/content/drive/MyDrive/Dataset/EXT4_XML_Dataset/ext4_CCD/correct
/content/drive/MyDrive/Dataset/EXT4_XML_Dataset/FewShots
/content/drive/MyDrive/Dataset/EXT4_XML_Dataset/FewShots/ext4_CPD
/content/drive/MyDrive/Dataset/EXT4_XML_Dataset/FewShots/ext4_CPD/ValidConfig
/content/drive/MyDrive/Dataset/EXT4_XML_Dataset/FewShots/ext4_CPD/Misconfig
/content/drive/MyDrive/Dataset/EXT4_XML_Dataset/FewShots/ext4_SD
/content/drive/MyDrive/Dataset/EXT4_XML_Dataset/FewShots/ext4_SD/ValidConfig
/content/drive/MyDrive/Dataset/EXT4_XML_Dataset/FewShots/ext4_SD/Misconfig
/content/drive/MyDri

In [6]:
DRIVE_DATA = "/content/drive/MyDrive/Dataset/EXT4_XML_Dataset"

In [7]:
!cp -r "$DRIVE_DATA/ext4_SD" icse25_data/datasets/synthesize_config/
!cp -r "$DRIVE_DATA/ext4_CPD" icse25_data/datasets/synthesize_config/
!cp -r "$DRIVE_DATA/ext4_CCD" icse25_data/datasets/synthesize_config/

!mkdir -p icse25_data/datasets/synthesize_config/FewShots

!cp -r "$DRIVE_DATA/FewShots/ext4_SD" \
icse25_data/datasets/synthesize_config/FewShots/

!cp -r "$DRIVE_DATA/FewShots/ext4_CPD" \
icse25_data/datasets/synthesize_config/FewShots/

!cp -r "$DRIVE_DATA/FewShots/ext4_CCD" \
icse25_data/datasets/synthesize_config/FewShots/

!find icse25_data/datasets/synthesize_config/FewShots -maxdepth 2

icse25_data/datasets/synthesize_config/FewShots
icse25_data/datasets/synthesize_config/FewShots/ext4_CPD
icse25_data/datasets/synthesize_config/FewShots/ext4_CPD/Misconfig
icse25_data/datasets/synthesize_config/FewShots/ext4_CPD/ValidConfig
icse25_data/datasets/synthesize_config/FewShots/ext4_CCD
icse25_data/datasets/synthesize_config/FewShots/ext4_CCD/Misconfig
icse25_data/datasets/synthesize_config/FewShots/ext4_CCD/ValidConfig
icse25_data/datasets/synthesize_config/FewShots/ext4_SD
icse25_data/datasets/synthesize_config/FewShots/ext4_SD/Misconfig
icse25_data/datasets/synthesize_config/FewShots/ext4_SD/ValidConfig


In [8]:
!python -m ciri.ciri_eng \
    --input_path \
        icse25_data/datasets/synthesize_config/ext4_SD/erroneous \
    --output_path \
        icse25_data/results/synthesize_config/ext4_SD/Qwen2.5-Coder-7B-Instruct/few_shot/erroneous \
    --model Qwen2.5-Coder-7B-Instruct \
    --system ext4 \
    --version 1.47.0 \
    --validconfig_shot_num 1 \
    --misconfig_shot_num 3 \
    --shot_selection random \
    --file_format xml \
    --verbose

2026-08-05 18:46:02 - Ciri - INFO - Using device: CUDA
2026-08-05 18:46:02 - Ciri - INFO - Using dtype: torch.bfloat16
config.json: 100% 663/663 [00:00<00:00, 4.22MB/s]
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 67.6MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/4.88G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/10.3G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/10.3G [00:00<?, ?B/s]
Reconstructing (incomplete total...):  16% 2.45G/15.2G [00:28<02:58, 71.8MB/s, 8.88MB/s  ]
Reconstructing (incomplete total...):  21% 3.19G/15.2G [00:38<03:49, 52.5MB/s, 10.3MB/s  ]
Reconstructing (incomplete total...):  21% 3.21G/15.2G [00:38<03:39, 54.8MB/s, 8.74MB/s  ]
Reconstructing (incomplete total...):  22% 3.28G/15.2G [00:48<11:38, 17.1MB/s, 11.2MB/s  ]
Reconstructing (incomplete total...):  22% 3.31G/15.2

In [9]:
!python -m ciri.ciri_eng \
    --input_path \
        icse25_data/datasets/synthesize_config/ext4_SD/correct \
    --output_path \
        icse25_data/results/synthesize_config/ext4_SD/Qwen2.5-Coder-7B-Instruct/few_shot/correct \
    --model Qwen2.5-Coder-7B-Instruct \
    --system ext4 \
    --version 1.47.0 \
    --validconfig_shot_num 1 \
    --misconfig_shot_num 3 \
    --shot_selection random \
    --file_format xml \
    --verbose

2026-08-05 18:57:33 - Ciri - INFO - Using device: CUDA
2026-08-05 18:57:33 - Ciri - INFO - Using dtype: torch.bfloat16
Loading weights: 100% 339/339 [01:01<00:00,  5.53it/s]
2026-08-05 18:58:38 - Ciri - INFO - Model loaded successfully on CUDA!
2026-08-05 18:58:38 - Ciri - INFO - Selected misconfiguration shots: [5, 13, 4]
2026-08-05 18:58:38 - Ciri - INFO - Selected valid configuration shots: [1]
2026-08-05 18:58:38 - Ciri - INFO - [llm_gen] Using device: CUDA
2026-08-05 18:58:48 - Ciri - INFO - [Qwen Cost] call=1, input_tokens=1335, output_tokens=103, total_tokens=1438, generation_time_seconds=9.6353
2026-08-05 18:58:48 - Ciri - INFO - [Qwen Cost Cumulative] calls=1, input_tokens=1335, output_tokens=103, total_tokens=1438, generation_time_seconds=9.6353
2026-08-05 18:58:59 - Ciri - INFO - [Qwen Cost] call=2, input_tokens=1335, output_tokens=135, total_tokens=1470, generation_time_seconds=10.8673
2026-08-05 18:58:59 - Ciri - INFO - [Qwen Cost Cumulative] calls=2, input_tokens=2670, ou

In [10]:
!python icse25_data/script/result_parser.py \
    --project ext4_SD \
    --model Qwen2.5-Coder-7B-Instruct \
    --mode few_shot

[Ciri Result] on ext4_SD with Qwen2.5-Coder-7B-Instruct and few_shot mode
File-Level: Precision: 0.50, Recall: 0.80, Accuracy: 0.50, F1: 0.62
Param-Level: Precision: 0.29, Recall: 0.80, Accuracy: 0.86, F1: 0.42


CPD

In [11]:
!python -m ciri.ciri_eng \
    --input_path \
        icse25_data/datasets/synthesize_config/ext4_CPD/erroneous \
    --output_path \
        icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/erroneous \
    --model Qwen2.5-Coder-7B-Instruct \
    --system ext4 \
    --version 1.47.0 \
    --validconfig_shot_num 1 \
    --misconfig_shot_num 3 \
    --shot_selection random \
    --file_format xml \
    --verbose

2026-08-05 19:06:22 - Ciri - INFO - Using device: CUDA
2026-08-05 19:06:22 - Ciri - INFO - Using dtype: torch.bfloat16
Loading weights: 100% 339/339 [01:01<00:00,  5.54it/s]
2026-08-05 19:07:28 - Ciri - INFO - Model loaded successfully on CUDA!
2026-08-05 19:07:28 - Ciri - INFO - Selected misconfiguration shots: [10, 14, 6]
2026-08-05 19:07:28 - Ciri - INFO - Selected valid configuration shots: [5]
2026-08-05 19:07:28 - Ciri - INFO - [llm_gen] Using device: CUDA
2026-08-05 19:07:35 - Ciri - INFO - [Qwen Cost] call=1, input_tokens=1350, output_tokens=76, total_tokens=1426, generation_time_seconds=7.4567
2026-08-05 19:07:35 - Ciri - INFO - [Qwen Cost Cumulative] calls=1, input_tokens=1350, output_tokens=76, total_tokens=1426, generation_time_seconds=7.4567
2026-08-05 19:07:41 - Ciri - INFO - [Qwen Cost] call=2, input_tokens=1350, output_tokens=65, total_tokens=1415, generation_time_seconds=6.3287
2026-08-05 19:07:41 - Ciri - INFO - [Qwen Cost Cumulative] calls=2, input_tokens=2700, outpu

In [12]:
!python -m ciri.ciri_eng \
    --input_path \
        icse25_data/datasets/synthesize_config/ext4_CPD/correct \
    --output_path \
        icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/correct \
    --model Qwen2.5-Coder-7B-Instruct \
    --system ext4 \
    --version 1.47.0 \
    --validconfig_shot_num 1 \
    --misconfig_shot_num 3 \
    --shot_selection random \
    --file_format xml \
    --verbose

2026-08-05 19:11:23 - Ciri - INFO - Using device: CUDA
2026-08-05 19:11:23 - Ciri - INFO - Using dtype: torch.bfloat16
Loading weights: 100% 339/339 [01:01<00:00,  5.54it/s]
2026-08-05 19:12:29 - Ciri - INFO - Model loaded successfully on CUDA!
2026-08-05 19:12:29 - Ciri - INFO - Selected misconfiguration shots: [12, 14, 13]
2026-08-05 19:12:29 - Ciri - INFO - Selected valid configuration shots: [1]
2026-08-05 19:12:29 - Ciri - INFO - [llm_gen] Using device: CUDA
2026-08-05 19:12:36 - Ciri - INFO - [Qwen Cost] call=1, input_tokens=1318, output_tokens=75, total_tokens=1393, generation_time_seconds=7.6342
2026-08-05 19:12:36 - Ciri - INFO - [Qwen Cost Cumulative] calls=1, input_tokens=1318, output_tokens=75, total_tokens=1393, generation_time_seconds=7.6342
2026-08-05 19:12:43 - Ciri - INFO - [Qwen Cost] call=2, input_tokens=1318, output_tokens=75, total_tokens=1393, generation_time_seconds=6.8733
2026-08-05 19:12:43 - Ciri - INFO - [Qwen Cost Cumulative] calls=2, input_tokens=2636, outp

In [14]:
!find "$DRIVE_DATA" -type f -iname "ext4_CPD.tsv"

/content/drive/MyDrive/Dataset/EXT4_XML_Dataset/Groundtruth/ext4_CPD.tsv


In [15]:
!mkdir -p icse25_data/datasets/synthesize_config/ground_truth

!cp "$DRIVE_DATA/Groundtruth/ext4_SD.tsv" \
    icse25_data/datasets/synthesize_config/ground_truth/

!cp "$DRIVE_DATA/Groundtruth/ext4_CPD.tsv" \
    icse25_data/datasets/synthesize_config/ground_truth/

!cp "$DRIVE_DATA/Groundtruth/ext4_CCD.tsv" \
    icse25_data/datasets/synthesize_config/ground_truth/

In [16]:
!ls -lh icse25_data/datasets/synthesize_config/ground_truth

total 52K
-rw-r--r-- 1 root root 3.4K Aug  5 18:28 alluxio.tsv
-rw-r--r-- 1 root root  735 Aug  5 18:28 django.tsv
-rw-r--r-- 1 root root 1.4K Aug  5 18:28 etcd.tsv
-rw------- 1 root root 2.7K Aug  5 19:16 ext4_CCD.tsv
-rw------- 1 root root  293 Aug  5 19:16 ext4_CPD.tsv
-rw-r--r-- 1 root root  229 Aug  5 19:16 ext4_SD.tsv
-rw-r--r-- 1 root root 3.2K Aug  5 18:28 hbase.tsv
-rw-r--r-- 1 root root 3.7K Aug  5 18:28 hcommon.tsv
-rw-r--r-- 1 root root 4.0K Aug  5 18:28 hdfs.tsv
-rw-r--r-- 1 root root 1.6K Aug  5 18:28 postgresql.tsv
-rw-r--r-- 1 root root 1.9K Aug  5 18:28 redis.tsv
-rw-r--r-- 1 root root 2.9K Aug  5 18:28 yarn.tsv
-rw-r--r-- 1 root root 1.3K Aug  5 18:28 zookeeper.tsv


In [18]:
!find icse25_data/results/synthesize_config/ext4_CPD \
    -maxdepth 5 -type f | head -100

icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/erroneous/1.xml
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/erroneous/5.xml
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/erroneous/7.xml
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/erroneous/3.xml
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/erroneous/6.xml
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/erroneous/4.xml
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/erroneous/2.xml
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/correct/1.xml
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/correct/5.xml
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/correct/7.xml
icse25_data/results/synthe

In [19]:
!find icse25_data/results/synthesize_config \
    -type d | grep -E "Qwen|few_shot|ext4_CPD"

icse25_data/results/synthesize_config/ext4_CPD
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/erroneous
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/correct
icse25_data/results/synthesize_config/ext4_SD/Qwen2.5-Coder-7B-Instruct
icse25_data/results/synthesize_config/ext4_SD/Qwen2.5-Coder-7B-Instruct/few_shot
icse25_data/results/synthesize_config/ext4_SD/Qwen2.5-Coder-7B-Instruct/few_shot/erroneous
icse25_data/results/synthesize_config/ext4_SD/Qwen2.5-Coder-7B-Instruct/few_shot/correct


In [21]:
%%bash

for f in icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/correct/*.xml
do
    mv "$f" "${f%.xml}"
done

for f in icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/erroneous/*.xml
do
    mv "$f" "${f%.xml}"
done

In [22]:
!find icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot \
    -maxdepth 2 -type f | sort

icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/correct/1
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/correct/2
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/correct/3
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/correct/4
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/correct/5
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/correct/6
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/correct/7
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/erroneous/1
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/erroneous/2
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/few_shot/erroneous/3
icse25_data/results/synthesize_config/ext4_CPD/Qwen2.5-Coder-7B-Instruct/f

In [23]:
!python icse25_data/script/result_parser.py \
    --project ext4_CPD \
    --model Qwen2.5-Coder-7B-Instruct \
    --mode few_shot

[Ciri Result] on ext4_CPD with Qwen2.5-Coder-7B-Instruct and few_shot mode
File-Level: Precision: 0.56, Recall: 0.71, Accuracy: 0.57, F1: 0.63
Param-Level: Precision: 0.00, Recall: 0.00, Accuracy: 0.85, F1: N.A.


CCD

In [24]:
!python -m ciri.ciri_eng \
    --input_path \
        icse25_data/datasets/synthesize_config/ext4_CCD/erroneous \
    --output_path \
        icse25_data/results/synthesize_config/ext4_CCD/Qwen2.5-Coder-7B-Instruct/few_shot/erroneous \
    --model Qwen2.5-Coder-7B-Instruct \
    --system ext4 \
    --version 1.47.0 \
    --validconfig_shot_num 1 \
    --misconfig_shot_num 3 \
    --shot_selection random \
    --file_format xml \
    --verbose

2026-08-05 19:23:16 - Ciri - INFO - Using device: CUDA
2026-08-05 19:23:16 - Ciri - INFO - Using dtype: torch.bfloat16
Loading weights: 100% 339/339 [01:01<00:00,  5.54it/s]
2026-08-05 19:24:22 - Ciri - INFO - Model loaded successfully on CUDA!
2026-08-05 19:24:22 - Ciri - INFO - Selected misconfiguration shots: [10, 11, 1]
2026-08-05 19:24:22 - Ciri - INFO - Selected valid configuration shots: [5]
2026-08-05 19:24:22 - Ciri - INFO - [llm_gen] Using device: CUDA
2026-08-05 19:24:28 - Ciri - INFO - [Qwen Cost] call=1, input_tokens=1617, output_tokens=50, total_tokens=1667, generation_time_seconds=6.2551
2026-08-05 19:24:28 - Ciri - INFO - [Qwen Cost Cumulative] calls=1, input_tokens=1617, output_tokens=50, total_tokens=1667, generation_time_seconds=6.2551
2026-08-05 19:24:34 - Ciri - INFO - [Qwen Cost] call=2, input_tokens=1617, output_tokens=50, total_tokens=1667, generation_time_seconds=5.5887
2026-08-05 19:24:34 - Ciri - INFO - [Qwen Cost Cumulative] calls=2, input_tokens=3234, outpu

In [25]:
!python -m ciri.ciri_eng \
    --input_path \
        icse25_data/datasets/synthesize_config/ext4_CCD/correct \
    --output_path \
        icse25_data/results/synthesize_config/ext4_CCD/Qwen2.5-Coder-7B-Instruct/few_shot/correct \
    --model Qwen2.5-Coder-7B-Instruct \
    --system ext4 \
    --version 1.47.0 \
    --validconfig_shot_num 1 \
    --misconfig_shot_num 3 \
    --shot_selection random \
    --file_format xml \
    --verbose

2026-08-05 19:46:41 - Ciri - INFO - Using device: CUDA
2026-08-05 19:46:41 - Ciri - INFO - Using dtype: torch.bfloat16
Loading weights: 100% 339/339 [01:01<00:00,  5.51it/s]
2026-08-05 19:47:47 - Ciri - INFO - Model loaded successfully on CUDA!
2026-08-05 19:47:47 - Ciri - INFO - Selected misconfiguration shots: [10, 12, 8]
2026-08-05 19:47:47 - Ciri - INFO - Selected valid configuration shots: [2]
2026-08-05 19:47:47 - Ciri - INFO - [llm_gen] Using device: CUDA
2026-08-05 19:48:01 - Ciri - INFO - [Qwen Cost] call=1, input_tokens=1406, output_tokens=155, total_tokens=1561, generation_time_seconds=13.7992
2026-08-05 19:48:01 - Ciri - INFO - [Qwen Cost Cumulative] calls=1, input_tokens=1406, output_tokens=155, total_tokens=1561, generation_time_seconds=13.7992
2026-08-05 19:48:10 - Ciri - INFO - [Qwen Cost] call=2, input_tokens=1406, output_tokens=108, total_tokens=1514, generation_time_seconds=9.6738
2026-08-05 19:48:10 - Ciri - INFO - [Qwen Cost Cumulative] calls=2, input_tokens=2812, 

In [26]:
!python icse25_data/script/result_parser.py \
    --project ext4_CCD \
    --model Qwen2.5-Coder-7B-Instruct \
    --mode few_shot

[Ciri Result] on ext4_CCD with Qwen2.5-Coder-7B-Instruct and few_shot mode
File-Level: Precision: 0.55, Recall: 0.98, Accuracy: 0.60, F1: 0.71
Param-Level: Precision: 0.07, Recall: 0.25, Accuracy: 0.75, F1: 0.11
